In [1]:
# pip install xarray netcdf4

In [2]:
import xarray as xr
import numpy as np
import os
import glob
from functools import reduce

folder = "raw_data"
files = sorted(glob.glob(os.path.join(folder, "*.nc")))
print("Found", len(files), "files\n")

Found 9 files



In [3]:
# ------------------------- STEP 1: VARIABLE MAP -------------------------

print("VARIABLE MAP (file → variables):\n")

for f in files:
    ds = xr.open_dataset(f)
    vars_in_file = list(ds.data_vars.keys())
    print(f"{os.path.basename(f)}  →  {vars_in_file}")

VARIABLE MAP (file → variables):

chl.nc  →  ['chl']
currents.nc  →  ['uo', 'vo']
nutrients.nc  →  ['fe', 'no3', 'po4', 'si']
o2.nc  →  ['o2']
ph.nc  →  ['ph']
so.nc  →  ['so']
spco2.nc  →  ['spco2']
thetao.nc  →  ['thetao']
wo.nc  →  ['wo']


In [4]:
# ------------------------- STEP 2: PREPROCESS -------------------------

def preprocess(ds, filename):
    print(f"--- Preprocessing {filename} ---")

    # --------------------------------------------------
    # 0) ADD MISSING DEPTH FOR FILES LIKE spco2.nc
    # --------------------------------------------------
    if "depth" not in ds.coords:
        print(f"  -> depth missing. Adding depth = 0.5")
        ds = ds.expand_dims({"depth": [0.5]})

    # --------------------------------------------------
    # 1) ROUND COORDINATES
    # --------------------------------------------------
    ds = ds.assign_coords({
        "depth":     np.round(ds["depth"].values, 1),
        "latitude":  np.round(ds["latitude"].values, 2),
        "longitude": np.round(ds["longitude"].values, 2),
    })

    # TRUNCATE TIME TO DAY
    if "time" in ds.coords:
        new_time = np.array(ds["time"].values, dtype="datetime64[D]")
        ds = ds.assign_coords(time=new_time)

    # --------------------------------------------------
    # 2) CHECK FOR INVALID DEPTH VALUES (≠ 0.5)
    # --------------------------------------------------
    depth_values = ds["depth"].values
    invalid_depths = depth_values[depth_values != 0.5]

    if len(invalid_depths) > 0:
        print(f"  -> WARNING: Found {len(invalid_depths)} depth values not equal to 0.5:")
        print(f"       Invalid depths: {invalid_depths.tolist()}")
    else:
        print(f"  -> All depth values are valid (=0.5)")

    # --------------------------------------------------
    # 3) DETECT DUPLICATES
    # --------------------------------------------------
    df = ds.to_dataframe().reset_index()

    key_cols = ["time", "depth", "latitude", "longitude"]

    # Count duplicates
    dup_mask = df.duplicated(subset=key_cols, keep=False)
    dup_count = df[dup_mask].shape[0]

    print(f"  -> Duplicate coordinate entries after rounding: {dup_count}")

    # --------------------------------------------------
    # 4) AVERAGE OUT DUPLICATES
    # --------------------------------------------------
    df = df.groupby(key_cols).mean().reset_index()

    # Convert back to xarray
    ds_clean = df.set_index(key_cols).to_xarray()

    print(f"  -> After averaging, unique grid cells: {ds_clean.to_dataframe().shape[0]}")

    return ds_clean

In [5]:
# ------------------------- STEP 3: LOAD + PREPROCESS -------------------------

datasets = []
print("Preprocessing all files...")

for f in files:
    print("\n Opening:", os.path.basename(f))
    ds_clean = xr.open_dataset(f, chunks={"time": 1})

    ds_clean = preprocess(ds_clean, os.path.basename(f))

    datasets.append(ds_clean)

Preprocessing all files...

 Opening: chl.nc
--- Preprocessing chl.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 390949

 Opening: currents.nc
--- Preprocessing currents.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 2822841

 Opening: nutrients.nc
--- Preprocessing nutrients.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 390949

 Opening: o2.nc
--- Preprocessing o2.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 390949

 Opening: ph.nc
--- Preprocessing ph.nc ---
  -> All depth values are valid (=0.5)
  -> Duplicate coordinate entries after rounding: 0
  -> After averaging, unique grid cells: 390949

 Opening: so.nc
--- Pre

In [6]:
# ------------------------- STEP 4: INNER JOIN COORDS -------------------------

def intersect_coord(coord):
    arrs = [ds[coord].values for ds in datasets]
    common = reduce(lambda a, b: np.intersect1d(a, b, assume_unique=True), arrs)
    base = datasets[0][coord].values
    mask = np.isin(base, common)
    return base[mask]


print("\nComputing INNER JOIN on coordinates...")

common_time  = intersect_coord("time")
common_depth = intersect_coord("depth")
common_lat   = intersect_coord("latitude")
common_lon   = intersect_coord("longitude")

print("\nAFTER INNER JOIN:")
print("  time:", len(common_time))
print("  depth:", len(common_depth))
print("  latitude:", len(common_lat))
print("  longitude:", len(common_lon))

if (
    len(common_time) == 0 or
    len(common_depth) == 0 or
    len(common_lat) == 0 or
    len(common_lon) == 0
):
    raise RuntimeError("Inner join produced empty coordinates! Aborting.")


Computing INNER JOIN on coordinates...

AFTER INNER JOIN:
  time: 1557
  depth: 1
  latitude: 13
  longitude: 17


In [7]:
# ------------------------- STEP 5: SUBSET -------------------------

aligned = []
print("Subsetting every dataset to common coords...\n")

for i, ds in enumerate(datasets):
    fname = os.path.basename(files[i])
    print("Subsetting:", fname)
    ds2 = ds.sel(
        time=common_time,
        depth=common_depth,
        latitude=common_lat,
        longitude=common_lon,
        method=None
    )
    aligned.append(ds2)

Subsetting every dataset to common coords...

Subsetting: chl.nc
Subsetting: currents.nc
Subsetting: nutrients.nc
Subsetting: o2.nc
Subsetting: ph.nc
Subsetting: so.nc
Subsetting: spco2.nc
Subsetting: thetao.nc
Subsetting: wo.nc


In [8]:
# ------------------------- STEP 6: MERGE -------------------------

print("\nMerging all variables...\n")

merged = xr.merge(aligned, combine_attrs="override")

merged = merged.sortby("time")


Merging all variables...



In [9]:
# ------------------------- STEP 7: SAVE OUTPUT -------------------------

output = "data.nc"
print("Saving final merged dataset to:", output)

merged.to_netcdf(output)

print("\nDONE!")
print("Saved:", output)
print("\nFinal dataset summary:\n")
merged

Saving final merged dataset to: data.nc

DONE!
Saved: data.nc

Final dataset summary:



<xarray.Dataset> Size: 18MB
Dimensions:    (time: 1557, depth: 1, latitude: 13, longitude: 17)
Coordinates:
  * time       (time) datetime64[s] 12kB 2022-06-01 2022-06-02 ... 2026-09-04
  * depth      (depth) float32 4B 0.5
  * latitude   (latitude) float32 52B 20.0 20.25 20.5 20.75 ... 22.5 22.75 23.0
  * longitude  (longitude) float32 68B 87.0 87.25 87.5 87.75 ... 90.5 90.75 91.0
Data variables: (12/13)
    chl        (time, depth, latitude, longitude) float32 1MB 0.286 ... nan
    uo         (time, depth, latitude, longitude) float32 1MB 0.1433 ... nan
    vo         (time, depth, latitude, longitude) float32 1MB 0.1213 ... nan
    fe         (time, depth, latitude, longitude) float32 1MB 0.001278 ... nan
    no3        (time, depth, latitude, longitude) float32 1MB 0.1593 ... nan
    po4        (time, depth, latitude, longitude) float32 1MB 0.0007956 ... nan
    ...         ...
    o2         (time, depth, latitude, longitude) float32 1MB 199.4 ... nan
    ph         (time, depth, latitude, longitude) float32 1MB 8.024 ... nan
    so         (time, depth, latitude, longitude) float32 1MB 32.92 ... nan
    spco2      (time, depth, latitude, longitude) float32 1MB 40.01 ... nan
    thetao     (time, depth, latitude, longitude) float32 1MB 30.36 ... nan
    wo         (time, depth, latitude, longitude) float32 1MB 5.102e-07 ... nan

In [10]:
import xarray as xr
import pandas as pd

# Load merged dataset
merged = xr.open_dataset("data.nc")

# Convert to DataFrame
df = merged.to_dataframe().reset_index()

df

,time,depth,latitude,longitude,chl,uo,vo,fe,no3,po4,si,o2,ph,so,spco2,thetao,wo
0,2022-06-01,0.5,20.0,87.00,0.285966,0.143302,0.121338,0.001278,0.159255,0.000796,2.166951,199.420868,8.024398,32.915531,40.012962,30.360138,5.101815e-07
1,2022-06-01,0.5,20.0,87.25,0.198817,0.285533,0.187017,0.001213,0.101606,0.000353,2.187757,198.929932,8.025283,32.908985,40.026024,30.338413,7.922670e-07
2,2022-06-01,0.5,20.0,87.50,0.157684,0.528991,0.245835,0.001165,0.072593,0.000246,2.220604,198.510590,8.026939,32.663116,39.930134,30.338757,8.813758e-07
3,2022-06-01,0.5,20.0,87.75,0.139504,0.651719,0.226462,0.001127,0.055083,0.000191,2.244844,198.271683,8.028164,32.484989,39.835117,30.413069,-7.822748e-07
4,2022-06-01,0.5,20.0,88.00,0.133310,0.715622,0.208896,0.001118,0.053426,0.000180,2.251250,198.161453,8.028683,32.405869,39.793114,30.456890,1.493642e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344092,2026-09-04,0.5,23.0,90.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
344093,2026-09-04,0.5,23.0,90.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
344094,2026-09-04,0.5,23.0,90.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
344095,2026-09-04,0.5,23.0,90.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df = df.dropna(how="any")

df

,time,depth,latitude,longitude,chl,uo,vo,fe,no3,po4,si,o2,ph,so,spco2,thetao,wo
0,2022-06-01,0.5,20.00,87.00,0.285966,0.143302,0.121338,0.001278,0.159255,0.000796,2.166951,199.420868,8.024398,32.915531,40.012962,30.360138,5.101815e-07
1,2022-06-01,0.5,20.00,87.25,0.198817,0.285533,0.187017,0.001213,0.101606,0.000353,2.187757,198.929932,8.025283,32.908985,40.026024,30.338413,7.922670e-07
2,2022-06-01,0.5,20.00,87.50,0.157684,0.528991,0.245835,0.001165,0.072593,0.000246,2.220604,198.510590,8.026939,32.663116,39.930134,30.338757,8.813758e-07
3,2022-06-01,0.5,20.00,87.75,0.139504,0.651719,0.226462,0.001127,0.055083,0.000191,2.244844,198.271683,8.028164,32.484989,39.835117,30.413069,-7.822748e-07
4,2022-06-01,0.5,20.00,88.00,0.133310,0.715622,0.208896,0.001118,0.053426,0.000180,2.251250,198.161453,8.028683,32.405869,39.793114,30.456890,1.493642e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344011,2026-09-04,0.5,21.75,91.00,19.444538,0.027866,-0.039173,0.011185,107.448914,0.014751,57.435593,307.469543,8.489056,0.960763,17.505991,30.044863,-1.969716e-06
344026,2026-09-04,0.5,22.00,90.50,15.035765,-0.038854,-0.057805,0.011769,75.961624,0.015124,25.470245,354.431091,8.484352,1.133262,10.987530,30.865955,-5.601034e-06
344027,2026-09-04,0.5,22.00,90.75,21.503191,-0.057656,-0.029828,0.013585,97.111176,0.040555,32.907940,378.296417,8.394803,1.110514,19.193230,30.222700,-5.181917e-06
344045,2026-09-04,0.5,22.25,91.00,14.931638,-0.009338,-0.080337,0.017141,92.673759,0.666895,13.429396,365.471008,7.894576,1.262806,60.417263,30.354458,-3.088933e-06


In [12]:
# Check unique depth values
if "depth" in df.columns:
    unique_depths = df["depth"].unique()
    print("Unique depth values:", unique_depths)

    # If all depth values are exactly 0.5 → drop the whole column
    if len(unique_depths) == 1 and unique_depths[0] == 0.5:
        print("All depth values are 0.5 → dropping depth column.")
        df = df.drop(columns=["depth"])
    else:
        print("Depth column has multiple values → keeping it.")

df

Unique depth values: [0.5]
All depth values are 0.5 → dropping depth column.


,time,latitude,longitude,chl,uo,vo,fe,no3,po4,si,o2,ph,so,spco2,thetao,wo
0,2022-06-01,20.00,87.00,0.285966,0.143302,0.121338,0.001278,0.159255,0.000796,2.166951,199.420868,8.024398,32.915531,40.012962,30.360138,5.101815e-07
1,2022-06-01,20.00,87.25,0.198817,0.285533,0.187017,0.001213,0.101606,0.000353,2.187757,198.929932,8.025283,32.908985,40.026024,30.338413,7.922670e-07
2,2022-06-01,20.00,87.50,0.157684,0.528991,0.245835,0.001165,0.072593,0.000246,2.220604,198.510590,8.026939,32.663116,39.930134,30.338757,8.813758e-07
3,2022-06-01,20.00,87.75,0.139504,0.651719,0.226462,0.001127,0.055083,0.000191,2.244844,198.271683,8.028164,32.484989,39.835117,30.413069,-7.822748e-07
4,2022-06-01,20.00,88.00,0.133310,0.715622,0.208896,0.001118,0.053426,0.000180,2.251250,198.161453,8.028683,32.405869,39.793114,30.456890,1.493642e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344011,2026-09-04,21.75,91.00,19.444538,0.027866,-0.039173,0.011185,107.448914,0.014751,57.435593,307.469543,8.489056,0.960763,17.505991,30.044863,-1.969716e-06
344026,2026-09-04,22.00,90.50,15.035765,-0.038854,-0.057805,0.011769,75.961624,0.015124,25.470245,354.431091,8.484352,1.133262,10.987530,30.865955,-5.601034e-06
344027,2026-09-04,22.00,90.75,21.503191,-0.057656,-0.029828,0.013585,97.111176,0.040555,32.907940,378.296417,8.394803,1.110514,19.193230,30.222700,-5.181917e-06
344045,2026-09-04,22.25,91.00,14.931638,-0.009338,-0.080337,0.017141,92.673759,0.666895,13.429396,365.471008,7.894576,1.262806,60.417263,30.354458,-3.088933e-06


In [13]:
# ------------------------------------------------------------
# CHECK FOR DUPLICATES USING PRIMARY KEY (time, lat, lon)
# ------------------------------------------------------------
pk = ["time", "latitude", "longitude"]

# Detect duplicates
dup_mask = df.duplicated(subset=pk, keep=False)
duplicate_rows = df[dup_mask]

if len(duplicate_rows) > 0:
    print("\nDUPLICATES FOUND!")
    print("Number of duplicate rows:", len(duplicate_rows))
    print("\nSample duplicates:")
    print(duplicate_rows.head())

    # --------------------------------------------------------
    # OPTION A — AVERAGE duplicates (recommended)
    # --------------------------------------------------------
    print("\nAveraging duplicate rows based on primary key...")
    df = df.groupby(pk).mean().reset_index()

else:
    print("\nNo duplicates found. Primary key is unique.")


No duplicates found. Primary key is unique.


In [14]:
df.to_csv("data.csv", index=False)

In [15]:
# ------------------------------------------------------------
# Named date-range splits → splits/*.csv
# Edit PERIODS below. role="model" is used by training.ipynb
# (internal train/test happens inside that slice).
# Every other period (role="independent") is used by testing.ipynb.
# Rows that fall in an independent window are dropped from model data
# so they cannot leak into fitting.
# ------------------------------------------------------------
import json
from pathlib import Path

PERIODS = {
    "model": {
        "start": "2023-12-01",
        "end": "2026-02-28",
        "role": "model",
    },
    "IOD_negative": {
        "start": "2022-09-01",
        "end": "2022-11-30",
        "role": "independent",
    },
    "IOD_positive": {
        "start": "2023-09-01",
        "end": "2023-11-30",
        "role": "independent",
    },
}

SPLITS_DIR = Path("splits")
SPLITS_DIR.mkdir(exist_ok=True)

df_split = df.copy()
df_split["time"] = pd.to_datetime(df_split["time"], errors="coerce")

model_names = [n for n, p in PERIODS.items() if p.get("role") == "model"]
if len(model_names) != 1:
    raise ValueError("PERIODS must contain exactly one entry with role='model'")
MODEL_NAME = model_names[0]


def _in_range(frame, start, end):
    return (frame["time"] >= pd.Timestamp(start)) & (frame["time"] <= pd.Timestamp(end))


independent_mask = False
for name, spec in PERIODS.items():
    if spec.get("role") != "independent":
        continue
    independent_mask = independent_mask | _in_range(df_split, spec["start"], spec["end"])

bucket_meta = {}
for name, spec in PERIODS.items():
    mask = _in_range(df_split, spec["start"], spec["end"])
    if spec.get("role") == "model":
        n_before = int(mask.sum())
        n_overlap = int((mask & independent_mask).sum())
        mask = mask & ~independent_mask
        if n_overlap:
            print(f"{name}: dropped {n_overlap} rows that overlap independent periods ({n_before} → {int(mask.sum())})")
    part = df_split.loc[mask].sort_values("time").reset_index(drop=True)
    out_path = SPLITS_DIR / f"{name}.csv"
    part.to_csv(out_path, index=False)
    bucket_meta[name] = {
        "role": spec["role"],
        "start": spec["start"],
        "end": spec["end"],
        "n": int(len(part)),
        "path": str(out_path),
        "time_min": None if part.empty else str(part["time"].min().date()),
        "time_max": None if part.empty else str(part["time"].max().date()),
    }
    print(f"wrote {out_path}  n={len(part)}  role={spec['role']}")
    if part.empty:
        print(f"  WARNING: no rows in {name} for {spec['start']} → {spec['end']}")

split_meta = {
    "model_bucket": MODEL_NAME,
    "periods": PERIODS,
    "buckets": bucket_meta,
}
with open(SPLITS_DIR / "split_meta.json", "w") as f:
    json.dump(split_meta, f, indent=2)

print("wrote", SPLITS_DIR / "split_meta.json")

wrote splits/model.csv  n=105909  role=model
wrote splits/IOD_negative.csv  n=11739  role=independent
wrote splits/IOD_positive.csv  n=11739  role=independent
wrote splits/split_meta.json
